In [2]:
import pyspark.sql, pyspark.sql.functions
from pyspark.sql import SparkSession

# Стартиране на SparkSession 
spark = (
    SparkSession.builder
    .appName("NaiveBayesExample")
    .master("local[*]")
    .getOrCreate()
)
print("SparkSession стартиран успешно")
print("Spark версия:", spark.version)

SparkSession стартиран успешно
Spark версия: 4.0.1


In [3]:
# Зареждане на CSV в DataFrame
df = spark.read.csv('D:\my-repo\data\students-online.csv', header=True, inferSchema=True)
# Премахваме ID, ако не участва в модела
df = df.drop("ID")

In [4]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

# Кодиране на категорийни колони
gender_indexer = StringIndexer(inputCol="Gender", outputCol="GenderIndex", handleInvalid="keep")
degree_indexer = StringIndexer(inputCol="Degree", outputCol="DegreeIndex", handleInvalid="keep")
platform_indexer = StringIndexer(inputCol="PlatformUsage", outputCol="PlatformUsageIndex", handleInvalid="keep")
format_indexer = StringIndexer(inputCol="PreferredFormat", outputCol="PreferredFormatIndex", handleInvalid="keep")
# Кодиране на целевия клас
label_indexer = StringIndexer(inputCol="Satisfaction", outputCol="label", handleInvalid="keep")
# Обединяване на признаците в една колона features
assembler = VectorAssembler(
    inputCols=["GenderIndex","DegreeIndex","PlatformUsageIndex","PreferredFormatIndex","Age","PlatformHours","OutPlatformHours","Self-assessment"],
    outputCol="features")


In [7]:
# Наивен Бейсов класификатор
nb = NaiveBayes(
    labelCol="label",
    featuresCol="features",
    modelType="gaussian"   # за числови данни
)

# Pipeline
pipeline = Pipeline(stages=[
    gender_indexer,
    degree_indexer,
    platform_indexer,
    format_indexer,
    label_indexer,
    assembler,
    nb
])

# Разделяне на данните
train_df, test_df = df.randomSplit([0.7, 0.3], seed=42)

# Обучение
model = pipeline.fit(train_df)

# Прогнозиране
predictions = model.transform(test_df)

# Оценка
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

accuracy = evaluator_acc.evaluate(predictions)
f1 = evaluator_f1.evaluate(predictions)

print("Accuracy =", accuracy)
print("F1-score =", f1)

# spark.stop()

Accuracy = 0.5
F1-score = 0.375
